#### 1. Import Necessary Libaries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from xgboost import XGBClassifier

#### 2.Load Data

In [2]:
features_df = pd.read_csv("../data/raw/ecommerce_customer_features.csv")
churn_df = pd.read_csv("../data/raw/ecommerce_customer_targets.csv")

### 3.Merge Datasets

In [3]:
df = pd.merge(features_df, churn_df, on="Customer_ID")

print("Dataset shape:", df.shape)
print(df.head())


Dataset shape: (6000, 16)
                            Customer_ID  account_age_months  avg_order_value  \
0  0520df14-712d-4c69-a0c5-95a2e7dfc1ff                  46           164.96   
1  a4013b3f-0688-4096-a194-6074be8ffec8                   3            39.09   
2  eb870f2c-ed3d-4a21-a8ac-273fae69ea4f                  29            37.42   
3  a7433451-8ea9-428a-9d80-679c6963b39f                  35            62.64   
4  43f81935-49e3-44d3-94d1-5c4715738988                  39           113.03   

   total_orders  days_since_last_purchase  discount_usage_rate  return_rate  \
0            12                        17                0.243       0.1720   
1             4                         5                0.591       0.0808   
2             8                        47                0.212       0.1424   
3             9                         3                0.699       0.0128   
4             1                         7                0.382       0.0232   

   customer_suppor

#### 4. Drop unnecessary columns

In [4]:
if "Customer_ID" in df.columns:
    df = df.drop(columns=["Customer_ID"])

#### 5.Clean and encode categorical variables

In [5]:
df["churned"] = df["churned"].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1})

df["loyalty_member"] = (
    df["loyalty_member"].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1})
)

#### 6.Handle missing values

In [6]:
print("\nMissing values before handling:")
print(df.isnull().sum())



Missing values before handling:
account_age_months             0
avg_order_value                0
total_orders                   0
days_since_last_purchase       0
discount_usage_rate            0
return_rate                    0
customer_support_tickets       0
loyalty_member                 0
browsing_frequency_per_week    0
cart_abandonment_rate          0
product_review_score_avg       0
engagement_score               0
satisfaction_score             0
price_sensitivity_index        0
churned                        0
dtype: int64


#### 7.Train-test split

In [7]:
# Separate features and target
X = df.drop("churned", axis=1)
y = df["churned"]

# Spliting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#### 8.Logistic Regression(baseline)

In [8]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

#### 9.Logistic Regression(balanced)

In [9]:
log_balanced_model = LogisticRegression(max_iter=1000, class_weight="balanced")
log_balanced_model.fit(X_train, y_train)

y_pred_log_balanced = log_balanced_model.predict(X_test)

#### 10.Random Forest

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

#### 11.XGBoost Model

In [11]:
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    scale_pos_weight=5,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

#### 12.Evaluation Function

In [12]:
def evaluate_model(name, y_true, y_pred):
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1 Score:", f1_score(y_true, y_pred))
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred))

#### 13.Evalute All Models

In [13]:
evaluate_model("Logistic Regression", y_test, y_pred_log)
evaluate_model("Logistic Regression (Balanced)", y_test, y_pred_log_balanced)
evaluate_model("Random Forest", y_test, y_pred_rf)
evaluate_model("XGBoost", y_test, y_pred_xgb)


Logistic Regression
Accuracy: 0.9733333333333334
Precision: 0.9398907103825137
Recall: 0.8911917098445595
F1 Score: 0.9148936170212766

Classification Report:

              precision    recall  f1-score   support

           0       0.98      0.99      0.98      1007
           1       0.94      0.89      0.91       193

    accuracy                           0.97      1200
   macro avg       0.96      0.94      0.95      1200
weighted avg       0.97      0.97      0.97      1200


Logistic Regression (Balanced)
Accuracy: 0.9683333333333334
Precision: 0.8538812785388128
Recall: 0.9689119170984456
F1 Score: 0.9077669902912622

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      1007
           1       0.85      0.97      0.91       193

    accuracy                           0.97      1200
   macro avg       0.92      0.97      0.94      1200
weighted avg       0.97      0.97      0.97      1200


Random Fores

#### 14.Feature Importance(Random Forest)

In [14]:
feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
feature_importance = feature_importance.sort_values(ascending=False)

print("\nFeature Importance:")
print(feature_importance.head(10))


Feature Importance:
engagement_score               0.434092
days_since_last_purchase       0.392209
satisfaction_score             0.032009
browsing_frequency_per_week    0.020160
product_review_score_avg       0.019136
return_rate                    0.016090
cart_abandonment_rate          0.013418
avg_order_value                0.012498
discount_usage_rate            0.011761
account_age_months             0.011750
dtype: float64
